In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os



In [13]:
os.chdir(r'C:\WORK\IPC-HQ\RAAP - RISK ANALYSIS APPROACH\AFGHANISTAN\CLIMATOLOGICAL\pythonProject\data')



In [14]:
morbidity_ke = pd.read_excel('morbidity_kenya.xlsx')
morbidity_ke.head(5)



,province,time,malaria,diarrhoea,urti
0,Garissa,April 2019,6750.0,2398,6750.0
1,Isiolo,April 2019,3656.0,804,3656.0
2,Kajiado,April 2019,13581.0,4089,13581.0
3,Kilifi,April 2019,17036.0,5748,17036.0
4,Kitui,April 2019,15932.0,2897,15932.0


In [15]:
morbidity_ke = morbidity_ke.rename(columns={
    'province':'adm1_name',
    'malaria':'Malaria',
    'diarrhoea':'Diarrhoea',
    'urti':'URTI'
})

In [16]:
# ==========================================================
# CREATE DATE COLUMN FROM TIME
# ==========================================================

morbidity_ke['date'] = pd.to_datetime(
    morbidity_ke['time'],
    format='%B %Y'
)

# Check result
morbidity_ke[['time', 'date']].sample(10)

,time,date
1875,July 2021,2021-07-01
1960,June 2021,2021-06-01
89,April 2024,2024-04-01
2252,November 2020,2020-11-01
1868,July 2021,2021-07-01
500,January 2022,2022-01-01
2381,October 2023,2023-10-01
1888,July 2022,2022-07-01
139,August 2020,2020-08-01
489,January 2021,2021-01-01


In [17]:
morbidity_ke['country']='Kenya'

In [18]:
morbidity_ke = morbidity_ke[['country', 'date','adm1_name', 'Malaria', 'Diarrhoea','URTI']]

In [19]:
morbidity_ke_long = morbidity_ke.melt(id_vars=['country', 'adm1_name','date'],
                                     value_vars=['Malaria', 'Diarrhoea','URTI'],
                                     value_name='value',
                                     var_name='indicator') 

In [20]:
morbidity_ke_long.sample(5)

,country,adm1_name,date,indicator,value
366,Kenya,Wajir,2020-02-01,Malaria,4918.0
2228,Kenya,Kieni,2019-11-01,Malaria,1757.0
4372,Kenya,Mbeere,2020-07-01,Diarrhoea,548.0
1011,Kenya,Kilifi,2019-11-01,Malaria,15077.0
1974,Kenya,Baringo,2022-06-01,Malaria,355.0


In [21]:
# Counties to retain
counties = [
    "Baringo",
    "Embu",
    "Garissa",
    "Isiolo",
    "Kajiado",
    "Kilifi",
    "Kitui",
    "Kwale",
    "Laikipia",
    "Lamu",
    "Makueni",
    "Mandera",
    "Marsabit",
    "Meru",
    "Narok",
    "Nyeri",
    "Samburu",
    "Taita Taveta",
    "Tana River",
    "Tharaka Nithi",
    "Turkana",
    "Wajir",
    "West Pokot",
]

# Keep only these counties
morbidity_ke_long = morbidity_ke_long[
    morbidity_ke_long["adm1_name"].isin(counties)
].copy()

print(f"Rows remaining: {len(morbidity_ke):,}")
print(sorted(morbidity_ke_long["adm1_name"].unique()))

Rows remaining: 2,520
['Baringo', 'Garissa', 'Isiolo', 'Kajiado', 'Kilifi', 'Kitui', 'Kwale', 'Laikipia', 'Lamu', 'Makueni', 'Mandera', 'Meru', 'Narok', 'Samburu', 'Taita Taveta', 'Tana River', 'Tharaka Nithi', 'Turkana', 'Wajir', 'West Pokot']


In [22]:
# Group by county and indicator
morbidity_ke_long_county = (
    morbidity_ke_long
    .groupby(["country","adm1_name", "indicator","date"], as_index=False)["value"]
    .sum(min_count=1)   # Keeps NaN if all values in a group are NaN
)

print(morbidity_ke_long_county.sample(5))

     country   adm1_name  indicator       date    value
2870   Kenya        Meru    Malaria 2020-03-01  12153.0
4992   Kenya  West Pokot       URTI 2022-01-01   6377.0
2880   Kenya        Meru    Malaria 2021-01-01     34.0
250    Kenya     Baringo       URTI 2025-11-01   3769.0
1299   Kenya       Kitui  Diarrhoea 2022-04-01   1833.0


In [23]:
morbidity_ke_long_county.to_excel('morbidity_long_kenya.xlsx',index=False)